[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/classical_mops.ipynb)

# MISDA on classical multi-objective reference problems

This notebook applies MISDA to established DTLZ reference problems. Unlike the controlled `diagnostic_*` suite, these problems are **not** assigned MISDA-specific latent or structural ground truth. Known Pareto-front geometry is reported only as external context.

In [ ]:
from pathlib import Path
import subprocess
import sys

target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd
import misda
import misda.benchmarks as bench

N = 300
M = 10
N_VARS = 19
SEED = 123
ON_FRONT = True
PROBLEM_IDS = ("dtlz2", "dtlz5")

## Reference protocol

We sample reproducibly from the analytical Pareto fronts (`on_front=True`). For DTLZ2 the front is the positive-orthant unit hypersphere, with manifold dimension `M-1`. For DTLZ5 it collapses to a one-dimensional curve. These manifold dimensions describe the classical MOP geometry; they are **not** declarations of MISDA `latent_dimension` or `structural_dimension`.

In [ ]:
classical_results = {}
rows = []
for problem_id in PROBLEM_IDS:
    spec = bench.CLASSICAL_MOPS[problem_id]
    F, X = spec["generator"](
        N=N, M=M, n_vars=N_VARS, on_front=ON_FRONT, seed=SEED
    )
    frame = pd.DataFrame(F, columns=[f"f{i}" for i in range(1, M + 1)])
    mis_set = misda.discover(frame, name=spec["name"], seed=SEED)
    ranking = misda.rank(mis_set)
    misda.evaluate(
        mis_set, metrics=("linear", "pareto"), candidates=ranking[:1]
    )
    print(mis_set.report())
    mis_set.graph_plot(ranking=ranking)

    selected = ranking.selected
    classical_results[problem_id] = {
        "F": frame, "X": X, "result_obj": mis_set, "ranking": ranking
    }
    rows.append({
        "problem_id": problem_id,
        "name": spec["name"],
        "pareto_geometry": spec["pareto_geometry"],
        "pareto_manifold_dimension": spec["pareto_manifold_dimension"](M),
        "misda_latent": mis_set.analysis.latent_dimension,
        "misda_structural": mis_set.analysis.structural_dimension,
        "misda_selected": ranking.selected_dimension,
        "support": mis_set.support.status,
        "selected_mean_r2": selected.linear.mean_r2 if selected and selected.linear else None,
        "pareto_retention": selected.pareto.retention if selected and selected.pareto else None,
        "pareto_validity": selected.pareto.validity if selected and selected.pareto else None,
        "pareto_jaccard": selected.pareto.jaccard if selected and selected.pareto else None,
    })

classical_summary = pd.DataFrame(rows)

In [ ]:
classical_summary

## Interpretation boundary

The columns `misda_latent`, `misda_structural`, and `misda_selected` are observations produced by MISDA on these sampled objective sets. They should not be scored as errors against `pareto_manifold_dimension`: the latter is a geometric property of the analytical Pareto front and is not, by itself, the same estimand as either MISDA dimension.